# Irradiance proxy vs co-located pyranometers

The saturation analysis never had an irradiance measurement. It works from a **proxy**: `ref`, the
median of eleven reliable units' DC power, each divided by its own 99.9th percentile. That proxy is
dimensionless and says nothing absolute about the resource.

`irradiance_may2025_aug2026.csv` supplies two actual instruments covering the same site and
period:

| column | sensor | surface |
|---|---|---|
| `POA`  | PY3 | **top** --- front-side plane-of-array irradiance |
| `BPOA` | PY7 | **bottom** --- rear-side irradiance, the bifacial contribution |

Both channels belong to inverter `EN2`; the export also carries the pairing's own channel names in
its header, which is what fixes the `PY3`/`PY7` labels used throughout.

**Scope.** Read-only. Nothing in `sat_work/` is edited, no artefact is rewritten, and no threshold or
method constant is changed. The frozen harness (`bench.py`) and the canonical metric module
(`canon_metrics.py`) are imported exactly as committed, so every definition used below --- the mid
band, the irradiance gate, the decision domain, the control pairs, the deficit --- is the research's
own rather than a re-derivation.

> **Supersedes an earlier attempt.** A first version of this notebook used a file that contained a
> single irradiance column and no rear sensor. That export is gone; this notebook reads the corrected
> two-column file. The earlier conclusions about *scale* were wrong as a result, and are corrected
> here. The earlier conclusions about *method invariance* and *proxy noise* survive unchanged.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "sat_work" / "research"))

import bench as B                      # frozen regression harness / detector
import canon_metrics as CM             # the version lock's own metric definitions
from satsim import BAND, CONTROL_PAIRS, EU, PAIRS, RELIABLE, dq_mask

SENSOR_CSV = ROOT / "irradiance_may2025_aug2026.csv"

plt.rcParams.update({"figure.dpi": 115, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "axes.titlesize": 10,
                     "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
                     "legend.fontsize": 8})
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

print("proxy gate     : ref >=", B.REF_ON)
print("proxy mid band : ref in", BAND)
print("sensor file    :", SENSOR_CSV.name, f"({SENSOR_CSV.stat().st_size/1e6:.1f} MB)")

proxy gate     : ref >= 0.7
proxy mid band : ref in (0.35, 0.6)
sensor file    : irradiance_may2025_aug2026.csv (3.8 MB)


## 1 · The instruments, inspected before use

A raw logger export is not self-describing. This section establishes its geometry, finds the
artefacts, and records exactly what the cleaning step removes --- so later sections can be judged
against known inputs.

The two channels turn out to be useful **fault detectors** for each other: they sit on the same
structure, so a dropout in one that leaves the other sane is instrumentation, not weather. In this
export it is the *rear* channel that carries the artefacts while the front stays healthy --- the
reverse of the pairing this notebook originally read.

In [2]:
raw = pd.read_csv(SENSOR_CSV, encoding="utf-8-sig")
raw.columns = ["site_time", "poa", "bpoa"]      # PY3 (top) and PY7 (bottom)
raw["site_time"] = pd.to_datetime(raw["site_time"], format="%m-%d-%Y %H:%M:%S")
raw = raw.set_index("site_time").sort_index()
for c in ("poa", "bpoa"):
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

# The DST fall-back hour is written twice: every timestamp in 2025-11-02 01:00-01:55 appears
# once populated and once all-NaN.  A timestamp has to be unique before `reindex(grid)` can run,
# and an empty copy must never displace a measured one, so the collapse keeps the populated row
# of each pair rather than whichever copy the file happens to list first.
rows_delivered = len(raw)
repeated_ts = int(raw.index.duplicated().sum())
raw = (raw.assign(_reading=raw[["poa", "bpoa"]].notna().any(axis=1))
          .sort_values("_reading", kind="stable")
          .loc[lambda d: ~d.index.duplicated(keep="last")]
          .drop(columns="_reading")
          .sort_index())
rows_dropped = rows_delivered - len(raw)

grid = pd.date_range(raw.index.min(), raw.index.max(), freq="5min")
absent = grid.difference(raw.index)
sentinel = raw["poa"] <= -1e6
glitch = (raw["poa"] < -50) & ~sentinel
small_neg = (raw["poa"] < 0) & ~sentinel & ~glitch
rear_glitch = raw["bpoa"] < -50

# The sentinel note is built conditionally: this export contains none, and indexing an empty
# selector would raise rather than report.  The row still has to appear, because "zero sentinels"
# is a property of the file and not something a reader should have to infer from a missing line.
sentinel_note = ("none: the front channel carries no out-of-range code in this export"
                 if not sentinel.any()
                 else f"{raw.index[sentinel][0]:%Y-%m-%d %H:%M}, code -2^31")

display(pd.DataFrame([
    ("rows as delivered", f"{rows_delivered:,}", "one row per 5 minutes, night included"),
    ("duplicated timestamps", f"{repeated_ts:,}", "the DST fall-back hour 2025-11-02 01:00-01:55, written twice"),
    ("rows after collapsing them", f"{len(raw):,}", f"{rows_dropped} dropped --- each dropped row was the empty copy of a timestamp that also carries a reading"),
    ("span", f"{raw.index.min():%Y-%m-%d %H:%M} to {raw.index.max():%Y-%m-%d %H:%M}", ""),
    ("5-min grid", f"{len(grid):,} positions, {len(absent)} absent", "the skipped spring-forward hour, 2026-03-08 02:00-02:55"),
    ("PY3 NaN", f"{int(raw['poa'].isna().sum()):,}", "2026-05 (626) and 2025-12 (227); no channel-specific outage"),
    ("PY7 NaN", f"{int(raw['bpoa'].isna().sum()):,}", "almost the same rows as PY3: 970 shared, 164 + 94 one-sided -- logger gaps"),
    ("PY3 sentinel", f"{int(sentinel.sum())}", sentinel_note),
    ("PY3 daytime glitch", f"{int(glitch.sum())}", "the front channel stays positive through every daylight row"),
    ("PY3 small negative", f"{int(small_neg.sum()):,}", f"median {raw['poa'][small_neg].median():.1f}, min {raw['poa'][small_neg].min():.1f} W/m2 --- night zero offset, kept"),
    ("PY7 negative", f"{int((raw['bpoa'] < 0).sum()):,}", f"min {raw['bpoa'].min():.0f} W/m2 --- the rear channel's zero drifts well below PY3's"),
    ("PY7 rear glitch", f"{int(rear_glitch.sum())}", "2025-12-25..28 (263), 2026-01-06..08 (58), 2026-05-15 13:35-15:15 (19) --- down to -410 W/m2 WHILE PY3 reads ~940 --- a rear-channel fault"),
    ("peak PY3", f"{raw['poa'].max():.0f} W/m2", "2026-06-08 12:20"),
    ("peak PY7", f"{raw['bpoa'].max():.0f} W/m2", "2026-02-20 12:05"),
], columns=["property", "value", "note"]).style.hide(axis="index"))

property,value,note
rows as delivered,"140,544","one row per 5 minutes, night included"
duplicated timestamps,12,"the DST fall-back hour 2025-11-02 01:00-01:55, written twice"
rows after collapsing them,"140,532",12 dropped --- each dropped row was the empty copy of a timestamp that also carries a reading
span,2025-05-01 00:00 to 2026-08-31 23:55,
5-min grid,"140,544 positions, 12 absent","the skipped spring-forward hour, 2026-03-08 02:00-02:55"
PY3 NaN,"1,134",2026-05 (626) and 2025-12 (227); no channel-specific outage
PY7 NaN,"1,064","almost the same rows as PY3: 970 shared, 164 + 94 one-sided -- logger gaps"
PY3 sentinel,0,none: the front channel carries no out-of-range code in this export
PY3 daytime glitch,0,the front channel stays positive through every daylight row
PY3 small negative,"46,498","median -2.0, min -15.0 W/m2 --- night zero offset, kept"


In [3]:
poa = raw["poa"].mask(sentinel | glitch).reindex(grid)
bpoa = raw["bpoa"].reindex(grid)

display(pd.DataFrame([
    ("grid rows", f"{len(poa):,}"),
    ("PY3 available", f"{int(poa.notna().sum()):,}  ({100*poa.notna().mean():.2f} %)"),
    ("PY7 available", f"{int(bpoa.notna().sum()):,}  ({100*bpoa.notna().mean():.2f} %)"),
    ("PY3 daylight rows (> 1 W/m2)", f"{int((poa > 1).sum()):,}"),
    ("PY3 > 700 W/m2", f"{int((poa > 700).sum()):,}"),
    ("PY3 > 1000 W/m2", f"{int((poa > 1000).sum()):,}"),
], columns=["property", "value"]).style.hide(axis="index"))

property,value
grid rows,"140,544"
PY3 available,"139,398 (99.18 %)"
PY7 available,"139,468 (99.23 %)"
PY3 daylight rows (> 1 W/m2),"73,939"
PY3 > 700 W/m2,"22,740"
PY3 > 1000 W/m2,"2,044"


In [4]:
fig, ax = plt.subplots(1, 3, figsize=(13.6, 3.3))

w = raw.loc["2026-05-15 12:30":"2026-05-15 16:00", ["poa", "bpoa"]]
ax[0].plot(w.index, w["poa"].values, lw=1.7, color="#dc2626", label="PY3  (top)")
ax[0].plot(w.index, w["bpoa"].values, lw=1.7, color="#2563eb", label="PY7  (bottom)")
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_title("(a) the rear-channel fault, with the front channel healthy")
ax[0].set_ylabel("irradiance  [W/m$^2$]")
ax[0].legend(loc="lower left")
ax[0].tick_params(axis="x", rotation=30)

day = poa[poa > 1]
ax[1].hist(day, bins=80, color="#0f766e", alpha=0.85)
for q, c in ((0.5, "#111827"), (0.999, "#dc2626")):
    ax[1].axvline(day.quantile(q), color=c, ls="--", lw=1.3, label=f"p{q*100:g} = {day.quantile(q):.0f}")
ax[1].set_yscale("log")
ax[1].set_title("(b) front-side POA distribution")
ax[1].set_xlabel("POA  [W/m$^2$]")
ax[1].set_ylabel("5-min samples (log)")
ax[1].legend()

ax[2].plot(poa.index, poa.values, lw=0.4, color="#dc2626", alpha=0.7, label="PY3 front")
ax[2].plot(bpoa.index, bpoa.values, lw=0.4, color="#2563eb", alpha=0.7, label="PY7 rear")
ax[2].set_title("(c) the whole record")
ax[2].set_ylabel("W/m$^2$")
ax[2].legend(loc="upper right", markerscale=4)

fig.suptitle("Two pyranometers — 140,544 rows, 2025-05-01 to 2026-08-31", y=1.04, fontsize=11)
fig.tight_layout()
plt.show()

C:\Users\nhphuong\AppData\Local\Temp\ipykernel_5164\458417771.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2 · Joining the two records

The power record holds only daylight-adjacent rows (06:15--20:20 in this export); the sensor holds the
full 5-minute grid. The join is one-directional: every power row wants a sensor value.

Alignment is checked rather than assumed. A timestamp offset between the two loggers would show up as a
peak in the cross-correlation at a non-zero lag, and would silently degrade everything after it.

In [5]:
df = B.raw_df()          # 76,933 x 24, the canonical input
fe = B.raw_fe()          # preprocess() output: ref, active, DQ, dayidx
ref = fe["ref"]
poa_p = poa.reindex(df.index)
bpoa_p = bpoa.reindex(df.index)

display(pd.DataFrame([
    ("power rows", f"{len(df):,}"),
    ("with a PY3 value", f"{int(poa_p.notna().sum()):,}  ({100*poa_p.notna().mean():.3f} %)"),
    ("with a PY7 value", f"{int(bpoa_p.notna().sum()):,}  ({100*bpoa_p.notna().mean():.3f} %)"),
], columns=["property", "value"]).style.hide(axis="index"))


def corr_at_lag(k):
    s = poa.shift(k).reindex(df.index)
    m = (ref >= 0.5) & s.notna()
    return round(ref[m].corr(s[m]), 4)


lag = pd.DataFrame([(k, corr_at_lag(k)) for k in range(-3, 4)],
                   columns=["lag (5-min steps)", "corr(ref, PY3)"])
display(lag.style.hide(axis="index"))
print("best lag:", int(lag.loc[lag['corr(ref, PY3)'].idxmax(), 'lag (5-min steps)']),
      "-> the loggers share a clock; there is no time offset to correct")

property,value
power rows,"76,933"
with a PY3 value,"76,840 (99.879 %)"
with a PY7 value,"76,895 (99.951 %)"


lag (5-min steps),"corr(ref, PY3)"
-3,0.605100
-2,0.646900
-1,0.745400
0,0.948900
1,0.705600
2,0.627200
3,0.593200


best lag: 0 -> the loggers share a clock; there is no time offset to correct


## 3 · Scale: what one `ref` unit is worth

`ref` is dimensionless but not arbitrary. Each unit is divided by its own 99.9th percentile of power,
so `ref` is proportional to power and therefore --- if the proxy is any good --- proportional to
irradiance.

The conversion constant is estimated the way the method estimates everything else: as a **median over
the mid band `ref ∈ [0.35, 0.60]`**, the band `theta_roll` uses to anchor the slope and the band in
which clipping cannot bind.

In [6]:
mid = (ref >= BAND[0]) & (ref <= BAND[1]) & (poa_p > 0)
k = float((ref[mid] / poa_p[mid]).median())          # ref per W/m2
W_PER_REF = 1.0 / k

display(pd.DataFrame([
    ("mid band lower   (ref = 0.35)", f"{BAND[0]:.2f}", f"{BAND[0]*W_PER_REF:6.0f} W/m2"),
    ("mid band upper   (ref = 0.60)", f"{BAND[1]:.2f}", f"{BAND[1]*W_PER_REF:6.0f} W/m2"),
    ("irradiance gate  (ref = 0.70)", f"{B.REF_ON:.2f}", f"{B.REF_ON*W_PER_REF:6.0f} W/m2"),
    ("full scale       (ref = 1.00)", "1.00", f"{W_PER_REF:6.0f} W/m2"),
    ("observed maximum", f"{ref.max():.3f}", f"{ref.max()*W_PER_REF:6.0f} W/m2"),
], columns=["method constant, in ref units", "ref", "implied front-side POA"]).style.hide(axis="index"))
print(f"1.00 ref unit  =  {W_PER_REF:.0f} W/m2 front-side POA     (k = ref/POA = {k:.6f}, n = {int(mid.sum()):,})")

stability = pd.DataFrame([
    ("mid band (the anchor)", 0.35, 0.60),
    ("high tail", 0.70, 0.85),
    ("top tail", 0.85, 1.05),
], columns=["window", "lo", "hi"])
vals = []
for _, r in stability.iterrows():
    s = (ref >= r["lo"]) & (ref < r["hi"]) & (poa_p > 0)
    vals.append(1.0 / (ref[s] / poa_p[s]).median())
stability["n"] = [int(((ref >= r["lo"]) & (ref < r["hi"]) & (poa_p > 0)).sum()) for _, r in stability.iterrows()]
stability["1 ref = W/m2"] = np.round(vals).astype(int)
stability["deviation"] = (100 * (np.array(vals) / W_PER_REF - 1)).round(1)
display(stability[["window", "n", "1 ref = W/m2", "deviation"]].style.hide(axis="index")
        .format({"deviation": "{:+.1f} %"}))

"method constant, in ref units",ref,implied front-side POA
mid band lower (ref = 0.35),0.35,372 W/m2
mid band upper (ref = 0.60),0.60,638 W/m2
irradiance gate (ref = 0.70),0.70,745 W/m2
full scale (ref = 1.00),1.00,1064 W/m2
observed maximum,1.074,1142 W/m2


1.00 ref unit  =  1064 W/m2 front-side POA     (k = ref/POA = 0.000940, n = 13,255)


window,n,1 ref = W/m2,deviation
mid band (the anchor),13255,1064,+0.0 %
high tail,13110,1114,+4.7 %
top tail,5972,1106,+4.0 %


In [7]:
D = pd.DataFrame({"ref": ref, "poa": poa_p})
D["month"] = df.index.to_period("M")
rows = []
for m, g in D.groupby("month"):
    mm = (g["ref"] >= BAND[0]) & (g["ref"] <= BAND[1]) & (g["poa"] > 0)
    if int(mm.sum()) < 200:
        continue
    rows.append((str(m), int(mm.sum()), 1.0 / (g["ref"][mm] / g["poa"][mm]).median()))
stab = pd.DataFrame(rows, columns=["month", "n (mid band)", "1 ref = W/m2"])
stab["deviation"] = (100 * (stab["1 ref = W/m2"] / W_PER_REF - 1)).round(1)
stab["1 ref = W/m2"] = stab["1 ref = W/m2"].round(0)
display(stab.style.hide(axis="index").format({"deviation": "{:+.1f} %"}))
print(f"month-to-month spread: {stab['deviation'].min():+.0f} % to {stab['deviation'].max():+.0f} %")
print("A sensor on a different plane would swing with solar geometry; this does not.")

month,n (mid band),1 ref = W/m2,deviation
2025-05,780,1034.000000,-2.8 %
2025-06,883,1058.000000,-0.6 %
2025-07,845,1089.000000,+2.4 %
2025-08,875,1072.000000,+0.7 %
2025-09,904,1083.000000,+1.8 %
2025-10,855,1103.000000,+3.7 %
2025-11,1184,1072.000000,+0.8 %
2025-12,719,989.000000,-7.0 %
2026-01,1014,1019.000000,-4.2 %
2026-02,839,1062.000000,-0.2 %


month-to-month spread: -7 % to +4 %
A sensor on a different plane would swing with solar geometry; this does not.


In [8]:
fig, ax = plt.subplots(1, 2, figsize=(12.4, 4.0))

m = (poa_p > 5) & ref.notna()
ax[0].hexbin(poa_p[m], ref[m], gridsize=70, bins="log", cmap="Blues", mincnt=1)
bins = np.arange(0, 1250, 50)
bi = np.digitize(poa_p, bins)
med = pd.Series(ref).groupby(bi).median()
cnt = pd.Series(ref).groupby(bi).size()
okb = (cnt >= 40) & (med.index >= 1)
ax[0].plot([bins[i - 1] for i in med.index[okb]], med[okb], "o-", color="#dc2626", ms=3.5, lw=1.6,
           label="binned median of ref")
xx = np.array([0, 1220])
ax[0].plot(xx, k * xx, "--", color="#111827", lw=1.2, label=f"ref = POA / {W_PER_REF:.0f}")
ax[0].set_xlabel("front-side POA  [W/m$^2$]")
ax[0].set_ylabel("proxy  ref  [dimensionless]")
ax[0].set_title("(a) the proxy is proportional to measured irradiance")
ax[0].legend(loc="upper left")

ratio = (ref / poa_p).groupby(bi).median()
okr = (cnt >= 40) & (ratio.index >= 1)
xs = np.array([bins[i - 1] for i in ratio.index[okr]])
ys = 100 * (ratio[okr] / k - 1)
ax[1].axhline(0, color="#111827", lw=1.1)
ax[1].bar(xs, ys, width=45, color=np.where(np.abs(ys) < 6, "#0f766e", "#f59e0b"))
ax[1].axvline(B.REF_ON * W_PER_REF, color="#dc2626", ls="--", lw=1.3,
              label=f"detector gate = {B.REF_ON*W_PER_REF:.0f} W/m$^2$")
ax[1].set_xlabel("front-side POA  [W/m$^2$]")
ax[1].set_ylabel("ref / POA  vs  band median  [%]")
ax[1].set_title("(b) linearity across the full operating range")
ax[1].legend()

fig.tight_layout()
plt.show()
print(f"over the band the detector actually uses "
      f"({BAND[0]*W_PER_REF:.0f}-{BAND[1]*W_PER_REF:.0f} W/m2), the ratio is flat.")

over the band the detector actually uses (372-638 W/m2), the ratio is flat.


C:\Users\nhphuong\AppData\Local\Temp\ipykernel_5164\1920667494.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4 · Shape: is the proxy linear in irradiance?

Proportionality is the property that matters, because the detector's expectation is
$\theta(t)\cdot\mathrm{ref}(t)$. If `ref` were a *non-linear* function of irradiance, the deficit would
carry a resource-dependent bias that no amount of $\theta$ calibration could remove.

In [9]:
PB = np.arange(0, 1350, 100)
tab = pd.DataFrame({"poa": poa_p, "ref": ref}).dropna()
tab["bin"] = pd.cut(tab["poa"], PB)
t = tab.groupby("bin", observed=True).agg(
    n=("ref", "size"), median_poa=("poa", "median"), median_ref=("ref", "median"),
    p10=("ref", lambda x: x.quantile(.10)), p90=("ref", lambda x: x.quantile(.90)))
t["1 ref = W/m2"] = (t["median_poa"] / t["median_ref"]).round(0)
t["vs band [%]"] = (100 * (t["median_poa"] / t["median_ref"] / W_PER_REF - 1)).round(1)
t["within-bin spread"] = (t["p90"] - t["p10"]).round(3)
display(t[["n", "median_poa", "median_ref", "within-bin spread", "vs band [%]"]]
        .rename(columns={"median_poa": "median POA", "median_ref": "median ref"})
        .style.hide(axis="index"))

sub = tab[tab["poa"].between(200, 1100)]
imp = sub['poa'].median() / sub['ref'].median()
print(f"over 200-1100 W/m2 (n = {len(sub):,}, {100*len(sub)/len(tab):.0f} % of the joined rows and "
      f"essentially the whole operating range) the implied constant is {imp:.0f} W/m2 per ref unit, "
      f"{100*(imp/W_PER_REF - 1):+.1f} % from the band-based value.")

n,median POA,median ref,within-bin spread,vs band [%]
17019,34.000000,0.024345,0.076000,31.300000
8136,146.000000,0.131330,0.093000,4.500000
6022,248.000000,0.230043,0.107000,1.300000
4839,350.000000,0.325265,0.123000,1.100000
4759,453.000000,0.421774,0.133000,1.000000
5232,552.000000,0.515723,0.131000,0.600000
5068,652.000000,0.600705,0.121000,2.000000
5442,753.000000,0.689465,0.126000,2.700000
6596,854.000000,0.772958,0.098000,3.900000
8638,950.000000,0.845649,0.097000,5.600000


over 200-1100 W/m2 (n = 48,589, 63 % of the joined rows and essentially the whole operating range) the implied constant is 1096 W/m2 per ref unit, +3.0 % from the band-based value.


## 5 · The array is bifacial, and the rear sensor measures it

The second column is not a duplicate. `BPOA` reads about 14 % of `POA` at midday --- the rear-side
irradiance a bifacial array collects off the ground and structure. Two things follow.

First, it is a **sanity check on the front channel**: the front channel stays positive at every
daylight row of this export, so the deep rear excursions recorded in section 1 can only be a fault in
the rear channel. `(POA > 0) & (BPOA < -50)` isolates 114 such rows --- all of 2025-12-25 to 28,
2026-01-06 to 08, and 2026-05-15.

Second, it raises a modeling question. A bifacial module's output is driven by front *and* rear
irradiance, so should the detector's proxy be front only, or front plus a weighted rear? The research's
`ref` is built from DC power, so it already contains whatever bifacial gain the array actually has ---
which means the test is empirical, not theoretical: does adding the rear term sharpen the control-pair
null?

In [10]:
day = (poa_p > 100) & (bpoa_p >= 0)
ratio_b = (bpoa_p / poa_p)[day]
by_hour = ratio_b.groupby(df.index[day].hour).median()

display(pd.DataFrame([
    ("median rear/front, daylight", f"{ratio_b.median():.4f}"),
    ("median rear/front, 11:00-13:00", f"{ratio_b[df.index[day].hour.isin([11, 12, 13])].median():.4f}"),
    ("rear peak", f"{bpoa.max():.0f} W/m2"),
    ("front peak", f"{poa.max():.0f} W/m2"),
], columns=["property", "value"]).style.hide(axis="index"))

fig, ax = plt.subplots(1, 2, figsize=(12.0, 3.6))
ax[0].plot(by_hour.index, by_hour.values, "o-", color="#2563eb", lw=1.8, ms=4)
ax[0].set_xlabel("hour of day")
ax[0].set_ylabel("median  rear / front")
ax[0].set_title("(a) bifacial ratio — peaked at midday, as expected")
ax[0].axhline(ratio_b.median(), color="#dc2626", ls="--", lw=1.2, label=f"median {ratio_b.median():.3f}")
ax[0].legend()

d0 = df.index[day][0]
sl = slice("2026-06-15 08:00", "2026-06-15 18:00")
ax[1].plot(poa.loc[sl].index, poa.loc[sl].values, lw=1.6, color="#dc2626", label="PY3 front")
ax[1].plot(bpoa.loc[sl].index, bpoa.loc[sl].values, lw=1.6, color="#2563eb", label="PY7 rear")
ax[1].set_ylabel("W/m$^2$")
ax[1].set_title("(b) front and rear on a clear day")
ax[1].legend()
ax[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

property,value
"median rear/front, daylight",0.1259
"median rear/front, 11:00-13:00",0.1401
rear peak,463 W/m2
front peak,1319 W/m2


C:\Users\nhphuong\AppData\Local\Temp\ipykernel_5164\148411425.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.1 The rear/front ratio

The cell above gives the by-hour curve. This one fixes the ratio's two properties that matter
downstream.

**It is not a constant.** It falls with front-side irradiance --- about $0.18$ at
100--200 W/m$^2$ down to a minimum near $0.11$ around 700, then drifts back up to about $0.14$ at the
top of the record --- because at low sun a larger share of the rear
sensor's view is diffuse sky and ground, whereas a clear high sun is dominated by the direct beam on
the front. A single figure like "14 %" is therefore meaningless without its operating point attached, and
the detector's mid-band and the clear-sky peak sit at different points on that curve.

**It is not stable across the record.** Compared at a *fixed* irradiance window --- which is the
point of that column: holding the sun constant stops winter from looking different merely because more
of its rows land in the low-irradiance bins above --- December 2025 roughly doubles and February 2026
sits about 15 % high, with 29 % of its fixed-window rows above $0.2$ against under 1 % for most months.
Neither is a sampling effect, so neither can be folded into one bifaciality constant.
December is consistent with a high-albedo ground surface (snow) lifting the rear channel; the
February tail has no such reading, and would need the site's ground-cover and cleaning records.

For the detector this is context, not an input: `ref` is built from DC power and so already carries the
array's real bifacial gain --- which is why the rear-term test at the end of this section is empirical
rather than theoretical.

In [11]:
REAR, FRONT = "PY7 rear (bottom)", "PY3 front (top)"
MIN_FRONT = 100.0                      # W/m2 -- below this a ratio is mostly division noise
FIXED_LOW, FIXED_HIGH = 600.0, 900.0   # the like-for-like window for the month comparison

ratio_full = bpoa_p / poa_p
ok = (poa_p > MIN_FRONT) & bpoa_p.notna() & (bpoa_p >= 0)
ratio = ratio_full[ok]
hour = df.index[ok].hour

display(pd.DataFrame([
    ("definition", f"{REAR} / {FRONT}"),
    ("rows used", f"{int(ok.sum()):,} of {len(df):,}   ({100 * ok.mean():.1f} %)"),
    ("cut", f"front > {MIN_FRONT:.0f} W/m$^2$"),
    ("median", f"{ratio.median():.4f}"),
    ("interquartile", f"{ratio.quantile(.25):.4f} -- {ratio.quantile(.75):.4f}"),
    ("5th -- 95th", f"{ratio.quantile(.05):.4f} -- {ratio.quantile(.95):.4f}"),
    ("midday median, 11--13 h", f"{ratio[hour.isin([11, 12, 13])].median():.4f}"),
], columns=["property", "value"]).style.hide(axis="index"))

# The ratio rises as the sun drops, so it is quoted against front-side irradiance.
B_ = pd.DataFrame({"front": poa_p[ok], "ratio": ratio})
B_["bin"] = pd.cut(B_["front"], np.arange(100, 1300, 100))
bt = B_.groupby("bin", observed=True).agg(
    n=("ratio", "size"), median=("ratio", "median"),
    p10=("ratio", lambda x: x.quantile(.10)), p90=("ratio", lambda x: x.quantile(.90)))
display(bt.round(4).style.hide(axis="index"))

# Month by month, and again at a FIXED irradiance window.  The fixed window is the honest
# comparison, and it is the column that exposes the two regimes below.
mon = df.index.to_period("M")
rows = []
for p in sorted(set(mon), key=str):
    sel = (mon == p) & ok
    inwin = sel & (poa_p > FIXED_LOW) & (poa_p <= FIXED_HIGH)
    rr, rw = ratio_full[sel], ratio_full[inwin]
    rows.append((str(p), int(sel.sum()), round(rr.median(), 4) if len(rr) else np.nan,
                 int(inwin.sum()), round(rw.median(), 4) if len(rw) else np.nan,
                 round(100 * (rw > 0.2).mean(), 1) if len(rw) else np.nan))
mt = pd.DataFrame(rows, columns=["month", "n (all daylight)", "median",
                                 f"n ({FIXED_LOW:.0f}--{FIXED_HIGH:.0f})",
                                 "median at fixed sun", "share > 0.2 [%]"])
display(mt.style.hide(axis="index"))

DEC, FEB = pd.Period("2025-12", "M"), pd.Period("2026-02", "M")
win = ok & (poa_p > FIXED_LOW) & (poa_p <= FIXED_HIGH)
ref_med = ratio_full[win & ~mon.isin([DEC, FEB])].median()
print(f"At a fixed {FIXED_LOW:.0f}-{FIXED_HIGH:.0f} W/m2 -- the same sun every month -- "
      f"two months break rank:")
for lbl, p in (("Dec 2025", DEC), ("Feb 2026", FEB)):
    r = ratio_full[win & (mon == p)]
    print(f"   {lbl} : median {r.median():.4f}  (n = {len(r):,})   vs {ref_med:.4f} for the "
          f"rest of the record   -> {100 * (r.median() / ref_med - 1):+.0f} %")
print()
print("Neither is a sampling effect, so the rear/front ratio is not one constant across this record.")
print("December is consistent with a high-albedo ground surface (snow) lifting the rear channel;")
print("February instead carries a long upper tail rather than a shifted median.  Nothing in this")
print("dataset confirms either cause -- both need the site's ground-cover and cleaning log.")

property,value
definition,PY7 rear (bottom) / PY3 front (top)
rows used,"56,652 of 76,933 (73.6 %)"
cut,front > 100 W/m$^2$
median,0.1259
interquartile,0.1003 -- 0.1574
5th -- 95th,0.0598 -- 0.2819
"midday median, 11--13 h",0.1401


n,median,p10,p90
8079,0.175400,0.080900,0.356100
6012,0.166700,0.072700,0.278600
4835,0.150100,0.062600,0.225000
4755,0.134600,0.057400,0.207200
5217,0.125500,0.058700,0.183200
5068,0.113800,0.065100,0.157900
5441,0.108500,0.072500,0.154000
6596,0.114800,0.083300,0.144900
8612,0.119700,0.098000,0.141500
1899,0.128500,0.110000,0.145200


month,n (all daylight),median,n (600--900),median at fixed sun,share > 0.2 [%]
2025-05,4208,0.122700,960,0.121500,0.600000
2025-06,4226,0.122100,1238,0.115300,0.000000
2025-07,4418,0.122900,1371,0.115900,0.000000
2025-08,4164,0.121200,1656,0.117800,0.100000
2025-09,3716,0.113600,1688,0.106300,0.000000
2025-10,2938,0.111300,1306,0.101700,0.000000
2025-11,2233,0.115400,345,0.096500,1.200000
2025-12,2130,0.338200,39,0.224400,69.200000
2026-01,2388,0.141800,270,0.106700,0.700000
2026-02,2786,0.150200,1180,0.128600,29.200000


At a fixed 600-900 W/m2 -- the same sun every month -- two months break rank:
   Dec 2025 : median 0.2244  (n = 39)   vs 0.1114 for the rest of the record   -> +101 %
   Feb 2026 : median 0.1286  (n = 1,180)   vs 0.1114 for the rest of the record   -> +15 %

Neither is a sampling effect, so the rear/front ratio is not one constant across this record.
December is consistent with a high-albedo ground surface (snow) lifting the rear channel;
February instead carries a long upper tail rather than a shifted median.  Nothing in this
dataset confirms either cause -- both need the site's ground-cover and cleaning log.


In [12]:
# Does folding the rear term into the proxy sharpen the error null? Test it on the control pairs.
BIFACIALITY = 1.0        # a rear photon is worth ~1 front photon at the module level
ref_front = (poa_p * k).clip(lower=0)
ref_bi = ((poa_p + bpoa_p * BIFACIALITY) * k).clip(lower=0)

rows = []
for a, b in CONTROL_PAIRS:
    d_ref = pd.Series(B.pair_deficit(df, fe, a, b)["deficit"])
    for lbl, r in (("array median (ref)", None), ("PY3 front only", ref_front), ("PY3 + PY7 rear", ref_bi)):
        if r is None:
            d, m = d_ref, CM.domain_mask(fe, a, b)
        else:
            fx = dict(fe); fx["ref"] = r
            d = pd.Series(B.pair_deficit(df, fx, a, b)["deficit"])
            m = CM.domain_mask(fx, a, b) & CM.domain_mask(fe, a, b)
        rows.append(dict(pair=f"{a}+{b}", proxy=lbl, sd=round(d[m].std(), 4)))
tab = pd.DataFrame(rows).pivot(index="pair", columns="proxy", values="sd")
display(tab.style.hide(axis="index"))
print("adding the rear term changes the null width by at most a few percent, and not in a consistent")
print("direction -- slightly worse on two of the three. The detector's DC-power proxy already contains")
print("the real bifacial gain, so an explicit rear term would be double counting it. The rear sensor's")
print("value here is diagnostic, not corrective.")

PY3 + PY7 rear,PY3 front only,array median (ref)
0.056300,0.052900,0.027700
0.055300,0.049700,0.042200
0.046200,0.043700,0.019400


adding the rear term changes the null width by at most a few percent, and not in a consistent
direction -- slightly worse on two of the three. The detector's DC-power proxy already contains
the real bifacial gain, so an explicit rear term would be double counting it. The rear sensor's
value here is diagnostic, not corrective.


## 6 · Would it have changed the detector?

### 6.1 What the method is, and is not, invariant to

Both detectors model the pair sum as a slope times the proxy:

$$\hat{s}(t)=\theta(t)\cdot\mathrm{ref}(t)\quad\text{(vat-v1)},\qquad
\hat{s}(t)=g_0\cdot c(t)\cdot\mathrm{ref}(t)\quad\text{(published)}$$

and both fix their slope by a median over a band. Rescale the proxy by $\alpha>0$ and the mid-band
median rescales by the same $\alpha$, so $\theta\to\theta/\alpha$ and

$$\theta'(t)\cdot\mathrm{ref}'(t)\;=\;\frac{\theta(t)}{\alpha}\cdot\alpha\,\mathrm{ref}(t)\;=\;\theta(t)\cdot\mathrm{ref}(t).$$

The expectation, the deficit, the gate and every flag are **exactly unchanged**.

But that invariance is conditional, and the condition matters here. The band `ref ∈ [0.35, 0.60]` and
the gate `ref ≥ 0.70` are **absolute numbers in `ref` units**. Rescaling the proxy without rescaling
them would move the band to a different irradiance, or off the data entirely, and change the answer.
So the correct statement is: *the method is invariant to a proxy's units provided the band and gate are
carried with it.* The constant from section 3 is exactly that carry.

In [13]:
def detect_in(df, fe, a, b, r, band, gate):
    '''vat-v1's rule with `band` and `gate` expressed in the units of the proxy `r`.

    The stock `B.det_rolling` hard-codes both in `ref` units, so it cannot express a proxy on a
    different scale. This is the same rule with those two constants made explicit.
    '''
    s = df[a] + df[b]
    act = fe["active"][a] & fe["active"][b]
    th = B.theta_roll(s, r, act, fe["dayidx"], q=0.5, band=band)
    d = (1 - s / (th * r)).replace([np.inf, -np.inf], np.nan)
    base = (r >= gate) & (d < 1 - B.NEAR_CAP) & act & ~dq_mask(fe, a, b)
    raw = base & (d > B.THR_MOD)
    return B.persist(B.bridge(raw, B.PERS_GAP, within=base), B.PERS_MOD, index=df.index), th, d


# (a) a PURE rescaling, with band and gate carried along -> the method must not move at all
rows = []
for alpha in (1.0, 2.0, 0.25, 1000.0):
    r2 = ref * alpha
    band2 = (BAND[0] * alpha, BAND[1] * alpha)
    same, er, dc = True, [], []
    for a, b in PAIRS:
        f0, th0, d0 = detect_in(df, fe, a, b, ref, BAND, B.REF_ON)
        f2, th2, d2 = detect_in(df, fe, a, b, r2, band2, B.REF_ON * alpha)
        same &= bool((np.asarray(f0, bool) == np.asarray(f2, bool)).all())
        m = np.isfinite(th0) & np.isfinite(th2)
        er.append(float(((th0 * ref)[m] / (th2 * r2)[m]).median()))
        dc.append(float(d0[m].corr(d2[m])))
    rows.append((alpha, same, min(er), min(dc)))
display(pd.DataFrame(rows, columns=["alpha  (ref -> alpha*ref)", "flags bit-identical",
                                    "min expectation ratio", "min deficit corr"])
        .style.hide(axis="index").format({"min expectation ratio": "{:.12f}",
                                          "min deficit corr": "{:.12f}"}))
print("A pure rescaling is a no-op: the flags are bit-identical and the expectation ratio is exactly 1.")
print()
print("The sensor is NOT a pure rescaling -- its ratio to ref varies row by row. That is what moves")
print("the answer, and it is the only thing that can.")

fe_s = dict(fe)                       # every other field of preprocess() is untouched
fe_s["ref"] = ref_front

rows = []
for a, b in PAIRS:
    f0, th0, d0 = detect_in(df, fe, a, b, ref, BAND, B.REF_ON)
    f2, th2, d2 = detect_in(df, fe_s, a, b, ref_front, BAND, B.REF_ON)
    m = (CM.domain_mask(fe, a, b) & CM.domain_mask(fe_s, a, b)
         & np.isfinite(th0) & np.isfinite(th2))
    f0n, f2n = np.asarray(f0, bool), np.asarray(f2, bool)
    rows.append((f"{a}+{b}", float(((th0 * ref)[m] / (th2 * ref_front)[m]).median()),
                 float(d0[m].corr(d2[m])), int(f0n.sum()), int(f2n.sum()),
                 round(100 * (f0n & f2n).sum() / max(f2n.sum(), 1), 1)))
display(pd.DataFrame(rows, columns=["pair", "expectation ratio", "deficit corr",
                                    "flags under ref", "flags under PY3", "% of PY3 flags shared"])
        .style.hide(axis="index").format({"expectation ratio": "{:.4f}", "deficit corr": "{:.4f}"}))
print("The expectation survives to ~1 % and the deficit correlates at 0.93-0.95: the signal is")
print("preserved. The gap is not a scale error -- it is the sensor's row-by-row disagreement with the")
print("array median, and it is what section 6.2 costs out.")

alpha (ref -> alpha*ref),flags bit-identical,min expectation ratio,min deficit corr
1.000000,True,1.000000000000,1.000000000000
2.000000,True,1.000000000000,1.000000000000
0.250000,True,1.000000000000,1.000000000000
1000.000000,True,1.000000000000,1.000000000000


A pure rescaling is a no-op: the flags are bit-identical and the expectation ratio is exactly 1.

The sensor is NOT a pure rescaling -- its ratio to ref varies row by row. That is what moves
the answer, and it is the only thing that can.


pair,expectation ratio,deficit corr,flags under ref,flags under PY3,% of PY3 flags shared
eu_8+eu_16,0.9912,0.9485,6584,7219,82.200000
eu_10+eu_18,0.9893,0.9525,8932,9837,84.500000
eu_13+eu_21,0.9880,0.9347,6075,6987,79.400000


The expectation survives to ~1 % and the deficit correlates at 0.93-0.95: the signal is
preserved. The gap is not a scale error -- it is the sensor's row-by-row disagreement with the
array median, and it is what section 6.2 costs out.


### 6.2 Reading the substitution

The **signal is unchanged**. On the shared pairs the two deficit series correlate at 0.93--0.95, and
the expectation ratio holds to 1.2 %, with the flags landing on the same rows. That is the invariance of
6.1 showing up empirically: whatever the
sensor reads, the pair sum is the same and the mid-band slope re-anchors.

The **noise is not**. A pyranometer is a single point sample of a cloud field that moves
across the array; the eleven-unit median integrates it. Every extra flag the sensor produces is a
false alarm --- it is a control pair, or a shared pair flagged on a row where nothing was constrained.
The control-pair total is the cleanest way to see it: 161 rows under `ref`, 335 under the sensor, with
`eu_1+eu_3` and `eu_4+eu_12` going from exactly zero to 90 and 56.

In [14]:
# Every detector is scored on ITS OWN decision domain, exactly as the research scores it: a flag is
# only ever set inside `act & ref >= gate & ~DQ`, and the two proxies do not define the same set.
rows = []
for a, b in PAIRS + CONTROL_PAIRS:
    _, f_ref = B.det_rolling(df, fe, a, b)
    _, f_sen = B.det_rolling(df, fe_s, a, b)
    f_ref = pd.Series(np.asarray(f_ref, bool), index=df.index)
    f_sen = pd.Series(np.asarray(f_sen, bool), index=df.index)
    both = CM.domain_mask(fe, a, b) & CM.domain_mask(fe_s, a, b)
    d_ref = pd.Series(B.pair_deficit(df, fe, a, b)["deficit"])
    d_sen = pd.Series(B.pair_deficit(df, fe_s, a, b)["deficit"])
    rows.append(dict(pair=f"{a}+{b}",
                     kind="shared" if (a, b) in PAIRS else "control",
                     flags_ref=int(f_ref.sum()), flags_sensor=int(f_sen.sum()),
                     agree=int((f_ref & f_sen & both).sum()),
                     ref_only=int((f_ref & ~f_sen & both).sum()),
                     sensor_only=int((~f_ref & f_sen & both).sum()),
                     deficit_corr=round(d_ref[both].corr(d_sen[both]), 3)))
sub = pd.DataFrame(rows)
display(sub.style.hide(axis="index"))

shared = sub[sub["kind"] == "shared"]
ctrl = sub[sub["kind"] == "control"]
print("flags on each detector's own decision domain")
print(f"  shared pairs : {shared['flags_ref'].sum():,} under ref  ->  {shared['flags_sensor'].sum():,} under PY3")
print(f"  control pairs: {ctrl['flags_ref'].sum():,} under ref  ->  {ctrl['flags_sensor'].sum():,} under PY3"
      "   <- every one of these is a false alarm")

pair,kind,flags_ref,flags_sensor,agree,ref_only,sensor_only,deficit_corr
eu_8+eu_16,shared,6584,7219,5932,549,1045,0.949000
eu_10+eu_18,shared,8932,9837,8308,557,1263,0.952000
eu_13+eu_21,shared,6075,6987,5547,472,1216,0.935000
eu_1+eu_3,control,0,90,0,0,76,0.602000
eu_4+eu_12,control,0,56,0,0,46,0.230000
eu_15+eu_23,control,161,189,88,65,74,0.581000


flags on each detector's own decision domain
  shared pairs : 21,591 under ref  ->  24,043 under PY3
  control pairs: 161 under ref  ->  335 under PY3   <- every one of these is a false alarm


In [15]:
nulls = []
for a, b in CONTROL_PAIRS:
    row = {"control pair": f"{a}+{b}"}
    for lbl, fex in (("ref", fe), ("PY3", fe_s)):
        d = pd.Series(B.pair_deficit(df, fex, a, b)["deficit"])
        m = CM.domain_mask(fex, a, b)
        row[f"sd ({lbl})"] = round(d[m].std(), 4)
        row[f"p1-p99 ({lbl})"] = round(d[m].quantile(.99) - d[m].quantile(.01), 3)
    row["ratio"] = round(row["sd (PY3)"] / row["sd (ref)"], 2)
    nulls.append(row)
display(pd.DataFrame(nulls).style.hide(axis="index"))
print("the control-pair null is 1.3-2.7x wider under the point sensor than under the array median")

control pair,sd (ref),p1-p99 (ref),sd (PY3),p1-p99 (PY3),ratio
eu_1+eu_3,0.027700,0.097000,0.059800,0.321000,2.160000
eu_4+eu_12,0.019400,0.092000,0.051400,0.281000,2.650000
eu_15+eu_23,0.042200,0.227000,0.057000,0.306000,1.350000


the control-pair null is 1.3-2.7x wider under the point sensor than under the array median


In [16]:
fig, ax = plt.subplots(1, 3, figsize=(13.6, 3.8))

a0, b0 = CONTROL_PAIRS[0]
for lbl, fex, col in (("ref (11-unit median)", fe, "#0f766e"), ("PY3 (point pyranometer)", fe_s, "#dc2626")):
    d = pd.Series(B.pair_deficit(df, fex, a0, b0)["deficit"])
    m = CM.domain_mask(fex, a0, b0)
    ax[0].hist(d[m], bins=120, range=(-0.3, 0.3), density=True, alpha=0.55, color=col, label=lbl)
ax[0].axvline(B.THR_MOD, color="k", ls="--", lw=1.1, label=f"flag threshold {B.THR_MOD}")
ax[0].set_title(f"(a) the error null, {a0}+{b0}\n(a control pair: cannot saturate)")
ax[0].set_xlabel("deficit  [dimensionless]")
ax[0].set_ylabel("density")
ax[0].legend(fontsize=7)

for j, (day_s, ttl) in enumerate([("2026-06-16", "(b) broken cloud"), ("2026-06-15", "(c) a clear stretch")]):
    a = ax[j + 1]
    sl = slice(day_s + " 09:00", day_s + " 17:00")
    a.plot(ref.loc[sl].index, ref.loc[sl].values, lw=1.7, color="#0f766e", label="ref  (array median)")
    a.plot(ref_front.loc[sl].index, ref_front.loc[sl].values, lw=1.3, color="#dc2626", alpha=0.85,
           label=f"PY3 / {W_PER_REF:.0f}")
    a.set_title(ttl)
    a.set_ylabel("irradiance, in ref units")
    a.legend(fontsize=7)
    a.tick_params(axis="x", rotation=30)

fig.suptitle("The proxy integrates over the array; the pyranometer samples one point", y=1.05, fontsize=11)
fig.tight_layout()
plt.show()

C:\Users\nhphuong\AppData\Local\Temp\ipykernel_5164\394937328.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.3 Two days, side by side, overlaid

Sections 6.1 and 6.2 make the argument in numbers. Here it is directly: the proxy and the co-located
pyranometer pair --- front (PY3) plus rear (PY7), the bifacial sensor plane --- drawn on a shared
time-of-day axis, one day per column. Set `DAYS` in the next code cell to any other dates; the sub-figure
captions are built from that same list, so they cannot end up naming a day other than the one plotted.

Each column has two rows that share an x-axis, and the two rows are two views of the same quantity. The
upper row overlays the two series. The lower row plots their difference, $\Delta(t) = r(t) - \hat{I}(t)$,
against zero --- the identical numbers, given an axis of their own so the sign and the size can be read
directly rather than by measuring a gap off a 0-1 scale. The two rows are on **different scales**: the
upper row spans the full $1.15$ of normalized range, the lower one only $\pm 0.12$, so the lower row is a
magnified view and any given difference looks roughly five times taller there. Amplitudes are comparable
within a row, never between the two.

Both rows draw the measured 5-minute record and nothing else. That is deliberate: the disagreement
between the two instruments **is** the subject of this section --- a cloud edge crosses one pyranometer
and not the other, and the eleven-unit median averages it away --- so a figure that averaged the
disagreement out before drawing it would be arguing against itself. How much of it survives a 60-minute
average is a question for the statistics cell below the figure, and that is where it is answered.

The sensor is put on the *proxy's* scale before it is drawn, because otherwise the overlay would measure a
definition rather than a disagreement. Each pyranometer samples one point of sky while `r(t)` is a median
over eleven units, so the summed `I(t)` sits a few percent below `r(t)` by construction --- a level that
carries no information about the resource. Dividing it out leaves a hatted sensor:

$$\hat{I}(t) = \frac{I(t)}{a}, \qquad a = \operatorname{median}_t \frac{I(t)}{r(t)} \text{ for that day}$$

$\hat{I}$ is `I(t)` expressed in the proxy's units, so the two curves coincide wherever the instruments
agree and separate only where they do not. Taking each day's own $a$ is also what lets the two columns be
read against each other, and --- because $\Delta$ is a difference of two already-aligned series --- what
makes the lower row center on zero: the definitional level was divided out before the subtraction, not
after.

Both series are normalized by the **same rule**: divide by their own $0.999$ quantile. That rule is also
the proxy's native unit --- it is what each unit is divided by inside `r(t)` --- and after it both series
sit at $1.000$ at their own $Q(0.999)$, which is what makes the two commensurable. So the y-axis of the
upper row is in those units, with $1$ at each series' own $Q(0.999)$ rather than at any physical
irradiance; the lower row inherits them, being a difference of the same two series.

The two days are **matched in level and opposite in character**, which is what lets the overlay be read as
a comparison rather than as two unrelated pictures. 2025-07-17 (a) is bright and broken: mean `ref`
$= 0.58$, peak $1.05$, and a third of its 5-minute steps move by more than $0.15$. 2026-06-03 (b) is at
least as bright on average --- mean $0.66$, median $0.86$, six rows in ten above $0.8$ --- but clean:
about one step in a hundred moves that far. Matching the levels matters because $\Delta$ is in absolute
units, so an overlay of two different irradiances would score the irradiance and not the instruments; a dim
second day would have made most of the contrast an artifact, and a bright clean one is the honest
counterpart to a bright broken one.

Days were not picked by eye. A screen over the 387 dates with usable daylight kept those in which the
proxy stays alive whenever the sensor sees sun (`ref > 0.15` while `POA > 300 W/m$^2$`); the days that
fail it are **proxy dropouts, not weather** --- `2026-07-23`, for instance, falls below `ref = 0.15` for
87 % of its sunny rows --- and plotting one would make the comparison a lie. That screen keeps a day's
*data* usable; it says nothing about how bright or how broken the sky was, so the two days here were then
picked from the days it kept, for the contrast above.

| day | character | corr | anchor `a` | residual \|Δ\| mean / p95 |
|---|---|---|---|---|
| **2025-07-17** (a) | bright and broken: mean `ref` 0.58; a third of the 5-min steps move > 0.15 | 0.994 | 0.98 | 0.023 / 0.063 |
| **2026-06-03** (b) | bright and clean: median `ref` 0.86; 60 % of rows above 0.8; 1 % of steps move > 0.15 | 0.998 | 0.97 | 0.014 / 0.039 |

The legend sits in the upper-right panel rather than the upper-left. Panel (a) has no empty region left
once the traces are drawn as measured --- its lower middle is filled by cloud dips, down to $0.35$ ---
whereas on 2026-06-03 neither trace drops below $0.84$ between 10 and 16 h, which is the span the key
occupies. The two series are distinguished by **line style rather than colour** throughout, because the
figure is set for grayscale printing: solid is `r(t)`, dashed is $\hat{I}(t)$. The lower row needs no key
of its own, carrying a single curve.


In [17]:
import os

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

DAYS = ["2025-07-17", "2026-06-03"]

# Two rows: the overlay on top, its residual underneath.  The figure is taller than the
# single-row version by the height of the residual row plus the gap above it, with the
# sub-captions still in a band at the foot.
FIG_W, FIG_H, DPI = 3.5, 2.6, 600

# Vector export for the paper.  Set SAVE_PDF True to also write the figure to disk at
# exactly FIG_W x FIG_H inches.  A PDF keeps the thin strokes and the LaTeX-set labels
# crisp at any zoom, which the raster PNG cannot; the PNG above stays the inline preview.
SAVE_PDF = True
PDF_PATH = "figures/irradiance_proxy_vs_sensor.pdf"

# LaTeX text, Computer Modern, 8 pt, grayscale only.
PLOT_RC = {
    "font.size": 8,
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],

    "axes.labelsize": 8,
    "axes.titlesize": 8,

    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.4,

    "axes.linewidth": 0.6,

    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,

    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,

    "axes.spines.top": False,
    "axes.spines.right": False,

    "legend.frameon": False,
    "legend.fontsize": 8,
}

# Line style.  The figure is set for grayscale, so the two series are told apart by
# STYLE rather than by colour: `r(t)` is solid and `I_hat(t)` is dashed, in the same
# ink and at the same weight.  Nothing is stacked by weight either, now that both are
# drawn from the measured record rather than from a smoothed version of it.
INK = "0.0"

TRACE_LW = 0.5

DASHES = (3.2, 2.0)

# The zero reference in the residual row: lighter than the data, so it reads as an axis
# rather than as a series, and dashed so it cannot be mistaken for one.
ZERO_COLOR = "0.55"
ZERO_LW = 0.5
ZERO_DASHES = (2.0, 2.0)

SHOW_LEGEND = True

# Sub-figure captions.  Placed below the axis label, so the figure needs extra height at
# the foot: the axes keep the height they had, and the new space is all bottom margin.
# One caption per COLUMN, under the residual row, since a caption names a day and both
# rows of a column are the same day.
CAPTION_PAD_PT = 27.0    # from the axes bottom down to the top of the caption

# ---------------------------------------------------------
# Grid geometry
# ---------------------------------------------------------

# Two equal-width columns and two rows, the overlay taller than the residual beneath it.
# `hspace` is a fraction of the average axes height, so 0.13 here is about 0.12 in of air
# between the rows -- enough to clear the tick marks on the shared edge, no more: the top
# row carries no x labels (they would otherwise be printed twice), so the space between
# the rows has nothing else to hold.
WSPACE = 0.06
HSPACE = 0.13
ROW_H_RATIO = [1.0, 0.75]

# Top row, in q0.999 units: each series sits at 1.0 at its own q0.999 by construction (see
# below).  The upper limit clears the tallest sample on either day (1.046, on 2025-07-17)
# without leaving a band of empty axes above it; the lower limit stays at 0 because both
# series are non-negative here.
Y_MIN, Y_MAX = 0.0, 1.15
Y_TICKS = [0.0, 0.5, 1.0]

# Bottom row: symmetric limits for the residual, so zero is centered and the sign is
# readable.  The largest excursion on either day is 0.112 (2025-07-17), which leaves the
# curve clear of the frame.
D_RESID = 0.12
D_TICKS = [-0.1, 0.0, 0.1]

X_TICKS = [6, 10, 14, 18]

# Moving-average window, in minutes, for the centered mean of `r(t)` and `I_hat(t)`.
# It is NOT drawn: the figure shows the measured 5-minute record alone, and this mean
# is used only by the moving-average statistics cell further down, which reports how
# much of each day's residual a 60-minute average would remove.  That is the quantity
# the figure used to show as a second, heavier pair of curves.
#
# It is applied on the clock rather than on a row count: the record is 5 min at best
# but not 5 min everywhere (section 2), and a row-count window would silently widen
# itself over a dropout.  It is centered, so the smoothed curve is not lagged against
# `t`; that makes it an analysis smoother rather than something the detector could
# have seen.
MA_WINDOW_MIN = 60.0


# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

# ---------------------------------------------------------
# The sensor, normalized for display
# ---------------------------------------------------------

# Both series are put on the same 0-1 scale for this figure by dividing by their own
# 0.999 quantile.  That is not an arbitrary choice: it is the rule that already defines
# `r(t)` -- each unit inside the proxy is divided by its own q0.999 before the median is
# taken -- so q0.999 is the proxy's native unit and q0.999(r) is 1 by construction.
#
# Using the SAME quantile on the sensor is what makes the two curves commensurable: after
# this step both series sit at 1.0000 at their own q0.999.  Dividing each by its own
# maximum instead would NOT: the pyranometer is far spikier than the 11-unit median
# (max/q0.999 is 1.175 against 1.076, because a cloud edge hits one sensor and is
# averaged away over the array), so a max rule rescales the two by different amounts and
# opens a spurious ~9.2% gap between them.
#
# This is a display choice only.  The detector's own sensor proxy (`ref_front`) stays on
# `ref`'s scale, in section 5, where the absolute band and gate apply.
#
# `I(t)` is the BIFACIAL sensor plane: the front (PY3) plane-of-array pyranometer PLUS the
# rear (PY7) one.  A rear photon is worth a front photon at the module level
# (BIFACIALITY = 1.0, section 5), so the two are summed rather than weighted.  A row with
# either sensor missing is left as NaN, so the `.dropna()` in `day_frame` drops it instead
# of letting a rear dropout read as a genuine irradiance shortfall.
sensor = poa_p + bpoa_p

SENSOR_Q999 = float(sensor.quantile(0.999))
REF_Q999 = float(ref.quantile(0.999))

i_norm = (sensor / SENSOR_Q999).clip(lower=0)
r_norm = ref / REF_Q999


def day_frame(day):
    """One day of the proxy, the scale-aligned sensor, their mean and their residual."""

    t0 = pd.Timestamp(day)

    # Include only the selected calendar day.
    t1 = t0 + pd.Timedelta(days=1)

    F = pd.DataFrame({
        "ref": r_norm.loc[t0:t1],
        "sensor": i_norm.loc[t0:t1],
        "poa": poa_p.loc[t0:t1],
    }).dropna()

    # Keep daytime measurements.
    F = F[F["poa"] > 5].copy()

    # Convert timestamps to time of day in hours.
    F["tod"] = (
        F.index.hour
        + F.index.minute / 60.0
    )

    # `I(t)` sits below `r(t)` because one is a point sensor and the other a median over
    # eleven units.  That level is definitional, not a disagreement, so it is divided out
    # before the two are overlaid, and the result is a hatted sensor:
    #
    #     I_hat(t) = I(t) / a        (a = that day's median ratio I/r)
    #
    # `I_hat` is `I(t)` put on the proxy's scale, so a gap between `r(t)` and `I_hat(t)` is
    # a disagreement rather than a definition.  Centering each day on its own `a` is what
    # makes the two panels comparable; without it each would merely restate its own anchor.
    F["anchor"] = float((F["sensor"] / F["ref"]).median())
    F["I_hat"] = F["sensor"] / F["anchor"]
    F["delta"] = F["ref"] - F["I_hat"]

    # Centered moving average of both series over `MA_WINDOW_MIN` of CLOCK time.  Rolling on
    # the timestamps rather than on the row count means the `poa > 5` filter above -- and
    # any logger dropout -- cannot widen the window: every point is a mean over a fixed span
    # of time, not over a fixed number of rows.
    win = f"{MA_WINDOW_MIN:g}min"

    F["ref_ma"] = F["ref"].rolling(win, center=True, min_periods=1).mean()
    F["I_hat_ma"] = F["I_hat"].rolling(win, center=True, min_periods=1).mean()

    # The mean is linear and `a` is a per-day constant, so this is identically the moving
    # average of `delta`: smoothing first and differencing after give the same curve.
    F["delta_ma"] = F["ref_ma"] - F["I_hat_ma"]

    return F


frames = [day_frame(d) for d in DAYS]


def subfigure_caption(day, letter):
    """'2025-07-17' -> '(a) July 17, 2025'.

    Derived from the same `DAYS` the panels are drawn from, so the caption cannot end up
    naming a day other than the one plotted.
    """
    return f"({letter}) {pd.Timestamp(day):%B %d, %Y}"


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

with matplotlib.rc_context(PLOT_RC):

    fig = plt.figure(
        figsize=(FIG_W, FIG_H),
        dpi=DPI,
    )

    # `sharey` within each row so the two days can be compared left to right, and `sharex`
    # within each column so the residual sits under the overlay it comes from.  The right
    # column carries no y-axis of its own and the top row carries no x labels.
    gs = fig.add_gridspec(
        2,
        2,
        width_ratios=[1, 1],
        height_ratios=ROW_H_RATIO,
        wspace=WSPACE,
        hspace=HSPACE,
    )

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1], sharey=ax1)

    ax3 = fig.add_subplot(gs[1, 0], sharex=ax1)
    ax4 = fig.add_subplot(gs[1, 1], sharex=ax2, sharey=ax3)

    # -----------------------------------------------------
    # Top row: the overlay
    # -----------------------------------------------------

    for ax, F in zip((ax1, ax2), frames):

        # The measured pair, drawn as measured.  `r(t)` is the array median over
        # eleven units; `I_hat(t)` is the summed front-plus-rear pyranometer put on the
        # proxy's scale.  Where the two coincide the instruments agree, and the gap
        # between them is the disagreement section 6.2 costs out -- which is the subject
        # of this figure, so it is deliberately not smoothed away here.  How much of it
        # a 60-minute average would remove is reported in the statistics cell below.
        ax.plot(
            F["tod"],
            F["ref"],
            color=INK,
            lw=TRACE_LW,
        )

        ax.plot(
            F["tod"],
            F["I_hat"],
            color=INK,
            lw=TRACE_LW,
            ls=(0, DASHES),
        )

    # -----------------------------------------------------
    # Bottom row: the residual
    # -----------------------------------------------------

    for ax, F in zip((ax3, ax4), frames):

        # Zero reference, so the sign of the residual is readable at a glance.
        ax.axhline(
            0.0,
            color=ZERO_COLOR,
            lw=ZERO_LW,
            ls=(0, ZERO_DASHES),
            zorder=1,
        )

        # The residual is `r(t) - I_hat(t)`: the same difference the row above shows as the
        # gap between its two curves, given its own axis so the sign and the size can be
        # read without measuring a gap off a 0-1 scale.  Negative means the sensor ran
        # ahead of the proxy.  It is differencing, not new data -- the top row and this one
        # are the two things plotted against each other, seen from different sides.
        ax.plot(
            F["tod"],
            F["delta"],
            color=INK,
            lw=TRACE_LW,
        )

    # -----------------------------------------------------
    # Axes
    # -----------------------------------------------------

    # Spans and ticks are set once per shared axis and inherited by the others.
    ax3.set_xlim(5, 21)
    ax4.set_xticks(X_TICKS)

    ax1.set_ylim(Y_MIN, Y_MAX)
    ax1.set_yticks(Y_TICKS)

    ax3.set_ylim(-D_RESID, D_RESID)
    ax3.set_yticks(D_TICKS)

    # The top row keeps its tick marks but not their labels: the bottom row carries the
    # shared axis, and with two rows the labels would otherwise be printed twice.
    for ax in (ax1, ax2):
        ax.tick_params(
            axis="x",
            labelbottom=False,
        )

    for ax in (ax3, ax4):
        ax.set_xlabel(
            r"$t$ (h)",
            labelpad=1.5,
        )

    # Symbols only: the axes are one line high each, and the legend in the upper-right
    # panel maps each symbol to its line style.  The definition of the hat is in the
    # prose above.
    ax1.set_ylabel(
        r"$r(t),\ \hat{I}(t)$",
        labelpad=1.5,
    )

    ax3.set_ylabel(
        r"$r(t)-\hat{I}(t)$",
        labelpad=1.5,
    )

    # Hide y-axis ticks and spine on the right column.  The two columns are close together
    # and share nothing in the middle, so dropping the inner spine lets each row read as one
    # panel cut at `t = 13` rather than as two.
    for ax in (ax2, ax4):
        ax.tick_params(
            axis="y",
            which="both",
            left=False,
            labelleft=False,
        )

        ax.spines["left"].set_visible(False)

    # -----------------------------------------------------
    # Legend
    # -----------------------------------------------------

    if SHOW_LEGEND:

        # One key per drawn series, in the upper-right panel.  It sits in panel (b), not
        # (a), because (b) is the day with a large empty region: on 2026-06-03 neither
        # trace drops below 0.84 between 10 and 16 h, which is exactly the span the key
        # occupies.  Panel (a) has no such window once the traces are drawn unsmoothed --
        # its lower middle is filled by the cloud dips, down to 0.35.
        handles = [
            Line2D(
                [], [],
                color=INK,
                lw=TRACE_LW,
            ),
            Line2D(
                [], [],
                color=INK,
                lw=TRACE_LW,
                ls=(0, DASHES),
            ),
        ]

        ax2.legend(
            handles,
            [r"$r(t)$", r"$\hat{I}(t)$"],
            loc="lower center",
            bbox_to_anchor=(0.5, 0.04),
            handlelength=1.9,
            handletextpad=0.4,
            labelspacing=0.3,
            borderaxespad=0,
        )

    # -----------------------------------------------------
    # Layout
    # -----------------------------------------------------

    # `left` clears the single-line ylabels and the signed tick labels ("-0.1" in the
    # residual row is the widest) and `right` is pinned to the frame.  `top` and `bottom`
    # fix the margins at 4 pt of ink at the top and at the top of the caption band at the
    # foot, so the two rows share 2.00 in of height: 1.08 in for the overlay, 0.12 in of
    # gap, 0.81 in for the residual.  The caption band below the residual row holds the x
    # tick labels, "t (h)" and the sub-captions.
    fig.subplots_adjust(
        left=0.145,
        right=0.995,
        top=0.9786,
        bottom=0.2087,
    )

    # -----------------------------------------------------
    # Sub-figure captions
    # -----------------------------------------------------

    for ax, day, letter in zip((ax3, ax4), DAYS, "ab"):

        # Anchored to the residual row's axes bottom and offset in points, so the caption
        # clears the tick labels and the axis label by a fixed amount at any figure size.
        ax.annotate(
            subfigure_caption(day, letter),
            xy=(0.5, 0.0),
            xycoords="axes fraction",
            xytext=(0, -CAPTION_PAD_PT),
            textcoords="offset points",
            ha="center",
            va="top",
        )

    # -----------------------------------------------------
    # Display at exact figure size in Jupyter
    # -----------------------------------------------------

    try:
        from matplotlib_inline.backend_inline import InlineBackend

        ib = InlineBackend.instance()

        prev = dict(ib.print_figure_kwargs)

        ib.print_figure_kwargs = {
            "bbox_inches": None,
            "pad_inches": 0,
        }

        try:
            plt.show()
        finally:
            ib.print_figure_kwargs = prev

    except ImportError:
        plt.show()

    # -----------------------------------------------------
    # Optional vector export
    # -----------------------------------------------------

    if SAVE_PDF:

        # `bbox_inches=None` writes the canvas as laid out, rather than cropping to the
        # artists.  Cropping would shave the margins you set in `subplots_adjust`, and
        # the file would no longer be FIG_W x FIG_H on the page.  Dropping CreationDate
        # keeps re-runs byte-stable, so an unchanged figure produces an unchanged file.
        os.makedirs(
            os.path.dirname(PDF_PATH) or ".",
            exist_ok=True,
        )

        fig.savefig(
            PDF_PATH,
            format="pdf",
            bbox_inches=None,
            pad_inches=0,
            metadata={"CreationDate": None},
        )

        print("saved:", PDF_PATH)


# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------

for day, F in zip(DAYS, frames):

    # `delta` already has the definitional level divided out, so the sign is meaningful
    # here: negative means the sensor ran ahead of the proxy.
    d = F["delta"]

    print(
        f"{day}: "
        f"rows {len(F)}, "
        f"corr {F['ref'].corr(F['sensor']):.3f}, "
        f"anchor a {F['anchor'].iloc[0]:.3f}, "
        f"|delta| mean {d.abs().mean():.3f}, "
        f"p95 {d.abs().quantile(0.95):.3f}, "
        f"rel p95 {(d.abs() / F['ref']).quantile(0.95):.3f}, "
        f"range {d.min():+.3f} to {d.max():+.3f}"
    )


C:\Users\nhphuong\AppData\Local\Temp\ipykernel_5164\3165444801.py:448: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: figures/irradiance_proxy_vs_sensor.pdf
2025-07-17: rows 173, corr 0.994, anchor a 0.979, |delta| mean 0.023, p95 0.063, rel p95 0.588, range -0.112 to +0.094
2026-06-03: rows 178, corr 0.998, anchor a 0.975, |delta| mean 0.014, p95 0.039, rel p95 0.274, range -0.025 to +0.055


In [18]:
# ---------------------------------------------------------
# Moving-average statistics
# ---------------------------------------------------------

# What the centered 60-minute mean does to the residual, per day.  The figure above
# draws the residual unsmoothed, so this is the only place the smoothing is quantified:
# it is what separates a disagreement that is merely fast from one that is persistent.
#
# `variance kept` is the share of the raw residual variance that survives smoothing: a
# disagreement was mostly high-frequency -- the point sensor tracking cloud that the array
# median averages away -- while a high share means a persistent disagreement that smoothing
# cannot remove.
for day, F in zip(DAYS, frames):

    d, dm = F["delta"], F["delta_ma"]

    # A nominal 60-minute window is 12 samples, but only where the grid is dense: `min_periods=1`
    # keeps the day's edges, and a dropout thins the middle.  So the window is described by the
    # samples it actually holds rather than by its nominal width.
    per_window = F["ref"].rolling(f"{MA_WINDOW_MIN:g}min").count()

    print(
        f"{day}: "
        f"window {MA_WINDOW_MIN:g} min "
        f"({per_window.median():.0f} samples median, {per_window.min():.0f} at the edges), "
        f"|delta| mean {d.abs().mean():.4f} -> {dm.abs().mean():.4f}, "
        f"p95 {d.abs().quantile(0.95):.3f} -> {dm.abs().quantile(0.95):.3f}, "
        f"pos/neg mean |delta| {d[d > 0].abs().mean():.4f}/{d[d < 0].abs().mean():.4f}, "
        f"range {dm.min():+.3f} to {dm.max():+.3f}, "
        f"variance kept {dm.var() / d.var():.1%}"
    )


2025-07-17: window 60 min (12 samples median, 1 at the edges), |delta| mean 0.0230 -> 0.0124, p95 0.063 -> 0.031, pos/neg mean |delta| 0.0231/0.0231, range -0.031 to +0.039, variance kept 26.4%
2026-06-03: window 60 min (12 samples median, 1 at the edges), |delta| mean 0.0142 -> 0.0133, p95 0.039 -> 0.038, pos/neg mean |delta| 0.0181/0.0102, range -0.017 to +0.042, variance kept 85.3%


**What to look at.** Read each column top to bottom. The upper row shows where the two instruments sit
relative to each other; the lower row shows how far apart they are, on an axis where zero is the middle
and is drawn as a light dashed rule. On 2026-06-03 (b) the two traces lie almost on top of each other and
the residual barely leaves its band: the largest excursion of the whole day is $0.055$. On 2025-07-17 (a)
the split repeats all through the middle of the day and the residual reaches $0.112$, twice as far. That
is a cloud edge crossing one pyranometer and not the other, with the eleven-unit median averaging it away
--- the effect section 6.2 costs out, here as a curve rather than as a histogram of flags.

The two days disagree in **kind**, not only in size, and the lower row alone cannot show that: both
residuals read as noise at this scale, and the one clear difference between them is amplitude. The
statistics cell below supplies the distinction. On (a) $|\Delta|$ averages $0.023$ and a centered
60-minute mean removes three quarters of its variance ($26\%$ kept), so the disagreement is fast --- which
is what cloud edges crossing a point sensor look like, and why the lower row of (a) is spiky. On (b)
$|\Delta|$ averages $0.014$ and the same mean removes almost none of it ($85\%$ kept), so what remains
there is slow, and (b)'s lower row is the shape of it: a curve that drifts rather than spikes.

That slow part is also asymmetric, which is the one feature of the lower row worth reading closely. Half
the samples sit on each side of zero on **both** days --- that is not a finding but an artifact of the
anchor, since $a$ is a median ratio and a median splits its sample in half --- so the count above the line
says nothing. The magnitudes do: on (b) the positive excursions average $0.018$ in absolute value against
$0.010$ for the negative ones, so when the two instruments disagree there, it is more often the proxy that
runs ahead. On (a) the same two figures are $0.023$ and $0.023$ --- dead symmetric --- so that day's
disagreement has no preferred direction, which is what independent cloud sampling should look like and is
consistent with it being the fast, weather-driven one.

The lower row is magnified roughly fivefold relative to the upper one --- $1.15$ of range against $0.24$ ---
so its excursions should not be compared against the apparent gap between the curves above them: the same
$0.06$ is a twentieth of the upper row's height and a quarter of the lower row's. Read each row against
its own axis. What the two rows share is the horizontal scale, which is the point of stacking them: a
spike in the residual sits directly under the moment in the upper row that produced it.

Because the days are matched in level --- within $14\%$ in mean `ref` --- the absolute residual is close to
a fair comparison, but $|\Delta|$ still scales with irradiance, so it is worth quoting the scale-free
version: the 95th percentile of $|\Delta|/r$ is $0.59$ on (a) against $0.27$ on (b), which keeps the
ordering. The broken day is the worse one both ways, which is the point.

Both are amplitude effects, not a timing offset. Cross-correlation at $\pm 3$ steps peaks at **lag zero**
on both days, so the two instruments share a clock and the proxy is not lagged against the sensor --- the
residual in the lower row is not the upper row shifted sideways.

None of this touches the detector. The deficit is $1 - s/(\theta r)$, so a constant multiplying the proxy
is absorbed by $\theta$ and the deficit itself is unchanged; only the band and the gate, being absolute in
`ref` units, would have to be carried across with it. Normalization is a display choice here --- which is
what lets `I(t)` be defined without consulting `r(t)`.


## 7 · The saturation edge, in absolute irradiance

The one thing the sensors can do that the proxy cannot: put a **physical number** on where the
constraint begins. The research's peer comparison --- each pair's output divided by an independent
pair's, normalized to 1 in the mid band --- is re-binned here by front-side POA instead of by `ref`,
converting the roll-off curve from proxy units into W/m$^2$.

In [19]:
X = df[EU].div(df[EU].quantile(0.999))
PEER = {("eu_8", "eu_16"): ("eu_1", "eu_3"), ("eu_10", "eu_18"): ("eu_1", "eu_3"),
        ("eu_13", "eu_21"): ("eu_1", "eu_3"), ("eu_1", "eu_3"): ("eu_4", "eu_12"),
        ("eu_4", "eu_12"): ("eu_1", "eu_3"), ("eu_15", "eu_23"): ("eu_4", "eu_12")}
edges = np.arange(0, 1250, 50)


def peer_curve(P, edges=edges):
    c, d = PEER[P]
    r = (X[P[0]] + X[P[1]]) / (X[c] + X[d]).replace(0, np.nan)
    ok = (r.notna() & (poa_p > 0) & fe["active"][P[0]] & fe["active"][P[1]]
          & fe["active"][c] & fe["active"][d])
    cal = ok & (poa_p >= BAND[0] * W_PER_REF) & (poa_p < BAND[1] * W_PER_REF)
    rn = r / r[cal].median()
    bi = np.digitize(poa_p, edges)
    return [(int(((bi == i) & ok).sum()),
             rn[(bi == i) & ok].median() if ((bi == i) & ok).sum() >= 25 else np.nan)
            for i in range(1, len(edges))]


def last_finite(out):
    return next((v for v in reversed([o[1] for o in out]) if np.isfinite(v)), np.nan)


gate_poa = B.REF_ON * W_PER_REF
fig, ax = plt.subplots(figsize=(10.0, 4.8))
for P, col in ((("eu_8", "eu_16"), "#dc2626"), (("eu_10", "eu_18"), "#f59e0b"), (("eu_13", "eu_21"), "#7c3aed")):
    out = peer_curve(P)
    xs = [edges[i - 1] + 25 for i in range(1, len(edges))]
    ax.plot(xs, [o[1] for o in out], "-", color=col, lw=2.0, marker="o", ms=3, label=f"{P[0]}+{P[1]}  (shared MPPT)")
for P, col in ((("eu_1", "eu_3"), "#0f766e"), (("eu_4", "eu_12"), "#2563eb"), (("eu_15", "eu_23"), "#64748b")):
    out = peer_curve(P)
    xs = [edges[i - 1] + 25 for i in range(1, len(edges))]
    ax.plot(xs, [o[1] for o in out], "--", color=col, lw=1.5, label=f"{P[0]}+{P[1]}  (independent)")

ax.axvline(gate_poa, color="#dc2626", ls=":", lw=1.7)
ax.text(gate_poa + 12, 1.16, f"detector gate\nref = {B.REF_ON:.2f}\n= {gate_poa:.0f} W/m$^2$",
        fontsize=8, color="#dc2626")
ax.axhline(1.0, color="#111827", lw=0.9)
ax.set_xlabel("front-side POA  [W/m$^2$]")
ax.set_ylabel("pair sum / independent peer    (1.0 = no constraint)")
ax.set_title("The MPPT constraint begins at ~800 W/m$^2$; the gate was already there")
ax.set_ylim(0.65, 1.28)
ax.set_xlim(0, 1250)
ax.legend(ncol=2, fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

C:\Users\nhphuong\AppData\Local\Temp\ipykernel_5164\3442720293.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
onset = []
for P in PAIRS:
    out = peer_curve(P)
    vals = [o[1] for o in out]
    hit = next((edges[i - 1] for i in range(1, len(edges))
                if np.isfinite(vals[i - 1]) and vals[i - 1] < 0.98), np.nan)
    tail = last_finite(out)
    onset.append((f"{P[0]}+{P[1]}", hit, round(tail, 3), f"{100 * (1 - tail):.0f} %"))
display(pd.DataFrame(onset, columns=["pair", "POA where the ratio first drops below 0.98",
                                     "ratio at the top bin", "shortfall at 1150-1200 W/m2"])
        .style.hide(axis="index"))

print("independent pairs at the top bin:",
      ", ".join(f"{a}+{b} {last_finite(peer_curve((a, b))):.3f}" for a, b in CONTROL_PAIRS))
print()
print(f"gate (ref >= {B.REF_ON})      = {gate_poa:.0f} W/m2")
print(f"observed onset, all 3 pairs = {onset[0][1]:.0f} W/m2")
print(f"the gate, chosen on a dimensionless proxy, lands {onset[0][1]-gate_poa:.0f} W/m2 below the "
      "irradiance at which the constraint actually binds.")

pair,POA where the ratio first drops below 0.98,ratio at the top bin,shortfall at 1150-1200 W/m2
eu_8+eu_16,800,0.637000,36 %
eu_10+eu_18,800,0.603000,40 %
eu_13+eu_21,800,0.651000,35 %


independent pairs at the top bin: eu_1+eu_3 0.972, eu_4+eu_12 1.029, eu_15+eu_23 0.987

gate (ref >= 0.7)      = 745 W/m2
observed onset, all 3 pairs = 800 W/m2
the gate, chosen on a dimensionless proxy, lands 55 W/m2 below the irradiance at which the constraint actually binds.


## 8 · The nameplate cross-check, re-read

`METHODS_RESULTS.md` §2.4 compares the detector's own slope against the documented **9.0 kW** string
rating and finds `median θ = 8,787 W`, a ratio of **0.976**. The sensor lets that comparison be read
more carefully, because it says what a `ref` of 1 actually *is* in W/m$^2$.

The comparison in the paper treats `θ` --- watts per `ref` unit --- as directly comparable to a
nameplate in watts. That is only exactly right if `ref = 1` coincides with the rating's irradiance. The
sensor says `ref = 1` is **well above** STC, so the two are different operating points, and the
agreement needs a temperature term to be interpreted.

In [21]:
th_u = {u: (df[u][(ref >= BAND[0]) & (ref <= BAND[1])] / ref[(ref >= BAND[0]) & (ref <= BAND[1])]).median()
        for u in RELIABLE}
theta_med = float(np.median(list(th_u.values())))
NAMEPLATE = 9000.0

display(pd.DataFrame([
    ("median mid-band slope, per unit (θ)", f"{theta_med:,.0f} W per ref unit"),
    ("what ref = 1 is, in irradiance", f"{W_PER_REF:,.0f} W/m2 front-side POA"),
    ("θ at that irradiance", f"{theta_med:,.0f} W"),
    ("θ normalized to STC 1000 W/m2", f"{theta_med*1000/W_PER_REF:,.0f} W"),
    ("ratio to the 9.0 kW nameplate, unadjusted", f"{theta_med*1000/W_PER_REF/NAMEPLATE:.3f}"),
], columns=["quantity", "value"]).style.hide(axis="index"))
print("METHODS_RESULTS.md quotes 8,787 W for the same quantity, on the 23-unit filter (all but eu_7);")
print("the 11-unit figure is 8,720 W. The 0.8 % difference is a filter choice, not a discrepancy.")

# A module at 1064 W/m2 is hot. Invert the usual temperature model:
#     P(G,T) ~ P_stc * (G/1000) * (1 + gamma*(T-25))
GAMMA = -0.0035      # /K, typical crystalline-silicon power coefficient
rows = []
for T in (45, 50, 55, 60, 65):
    p_stc = theta_med / ((W_PER_REF / 1000.0) * (1 + GAMMA * (T - 25)))
    rows.append((T, round(p_stc), round(p_stc / NAMEPLATE, 3)))
display(pd.DataFrame(rows, columns=["assumed cell temp [C]", "implied P_stc [W]", "P_stc / 9.0 kW"])
        .style.hide(axis="index"))
print(f"γ = {GAMMA}/K. The unadjusted comparison understates P_stc because ref = 1 sits "
      f"{100*(W_PER_REF/1000-1):.0f} % above STC irradiance;")
print("a normal module temperature over-corrects it back to within a few percent of the nameplate.")

quantity,value
"median mid-band slope, per unit (θ)","8,720 W per ref unit"
"what ref = 1 is, in irradiance","1,064 W/m2 front-side POA"
θ at that irradiance,"8,720 W"
θ normalized to STC 1000 W/m2,"8,197 W"
"ratio to the 9.0 kW nameplate, unadjusted",0.911


METHODS_RESULTS.md quotes 8,787 W for the same quantity, on the 23-unit filter (all but eu_7);
the 11-unit figure is 8,720 W. The 0.8 % difference is a filter choice, not a discrepancy.


assumed cell temp [C],implied P_stc [W],P_stc / 9.0 kW
45,8814,0.979000
50,8983,0.998000
55,9158,1.018000
60,9341,1.038000
65,9531,1.059000


γ = -0.0035/K. The unadjusted comparison understates P_stc because ref = 1 sits 6 % above STC irradiance;
a normal module temperature over-corrects it back to within a few percent of the nameplate.


**How to read this.** The paper's 0.976 was, in effect, comparing the array at `ref = 1`
(~1,064 W/m$^2$) against a rating defined at 1,000 W/m$^2$. Those are not the same operating point, and
the two effects that separate them point in opposite directions: the irradiance is **6 % above** STC,
which pushes the raw comparison down to ~0.91, while a hot module pushes it back up by roughly 8--16 %.
The near-unity agreement the paper reports is therefore **real but partly self-canceling**, and it
should be quoted with the operating point stated rather than as a bare ratio.

This does not weaken Validation 4 --- the array's peak per-unit output really does land near the rated
figure --- but it does mean the claim is a comparison of *observed peak* against *rated peak*, not an
STC normalization. **Recommended wording:** keep the 0.976 and put the operating point in the sentence
with it, e.g. *"θ = 8.8 kW per unit at ref = 1, which the co-located pyranometer places at ~1,064
W/m² — 6 % above STC, so the agreement is with the array's observed peak rather than an STC
normalization."*

## 9 · What this does and does not establish

**Established**

1. The proxy is a **faithful, linear measure of measured irradiance**. Over 200--1,100 W/m$^2$ ---
   63 % of the joined rows and essentially the whole operating range --- `ref` is proportional to
   front-side POA, and the constant holds to +4.7 % from the mid band to the top of the record. Month
   to month it varies by only **$-$7 % to $+$4 %**, which is what rules out a sensor on a materially
   different plane: one on a different tilt would swing with solar geometry by tens of percent.
2. The conversion is **1.00 `ref` unit $= 1{,}064$ W/m$^2$** of front-side POA. That gives the method's
   dimensionless constants a physical meaning for the first time: the mid band is **372--638 W/m$^2$**
   and the irradiance gate `ref ≥ 0.70` is **745 W/m$^2$**.
3. The detector's answer is **invariant to the units** of the proxy --- but conditionally, and the
   condition is worth stating: the band and the gate are absolute numbers in `ref` units, so a
   rescaling must carry them along. Verified exactly (`α` = 0.25, 2, 1000): flags **bit-identical**,
   expectation ratio **1.000000000000**, deficit correlation **1.0**.
4. The constraint is **located in absolute irradiance, and the gate was already on it**. All three
   shared-MPPT pairs track their independent peers to within 1 % up to 800 W/m$^2$, then leave them in
   the *same* 50 W/m$^2$ bin at **800 W/m$^2$**, reaching 35--40 % below peers by 1,150--1,200 W/m$^2$.
   The control pairs stay within 0.97--1.03 throughout. The `ref ≥ 0.70` gate, chosen on a dimensionless
   proxy with no knowledge of any irradiance measurement, sits **55 W/m$^2$ below** the point where the
   constraint actually binds.
5. The array is **bifacial**, and the rear pyranometer measures it: about 14 % of front at midday, peaking
   when the sun is high. Testing whether an explicit rear term sharpens the null says no --- the
   detector's DC-power proxy already contains the real bifacial gain, so adding it would double count.
   The rear sensor's contribution is diagnostic, not corrective: with the front channel healthy at
   $\sim$940 W/m$^2$, PY7 reads down to $-410$ W/m$^2$ across 19 rows of 2026-05-15, and again over
   2025-12-25 to 28 and 2026-01-06 to 08.
6. Substituting the front sensor does **not** move the saturation signal. The expectation survives to
   ~1 %, the two deficit series correlate at 0.93--0.95, and 79--85 % of the sensor's flags coincide
   with the proxy's.

**Not established, and better stated as limitations**

7. The pyranometer is **not a better proxy for this purpose, and would have been a worse one**. As a
   single point it does not average out the cloud field the way an eleven-unit median does. The
   control-pair null --- pure error, saturation impossible --- is 1.3--2.7$\times$ wider under the
   sensor, and substituting it manufactures false alarms the proxy does not: the control-pair total goes
   161 rows $\rightarrow$ 335, with 90 on `eu_1+eu_3` and 56 on `eu_4+eu_12`, both of which the proxy
   keeps at exactly zero. This is a property of *estimator geometry*, not of the sensor's accuracy.
8. The sensor is a point measurement, and its plane relative to the modules is not documented. The
   proportionality and onset results are robust to that (they depend on shape, not on absolute scale);
   the conversion in (2) inherits it, and an error there moves the W/m$^2$ numbers in sections 3 and 7
   together.
9. The largest logger gap in the joined record removes 59 front-side samples (2026-08), and the top of
   the roll-off curve rests on 38 rows above 1,150 W/m$^2$. The onset estimate is well sampled; the
   endpoint is not.
10. The nameplate agreement in §2.4 of `METHODS_RESULTS.md` is a comparison of **observed peak against
    rated peak**, not an STC normalization --- section 8 shows why those differ.
11. This is a **comparison only**. `ref` remains the canonical proxy, nothing was re-run, and the
    version lock was not touched.

**One item to settle with the data provider.** The absolute conversion in (2) depends on the
pyranometer's mounting plane and calibration. Asking for the tilt, azimuth, shading survey and last
calibration date would let the W/m$^2$ values in sections 3 and 7 be quoted as measured rather than as
inferred. Nothing else here depends on the answer.